In [16]:
from dotenv import load_dotenv

load_dotenv()

True

In [17]:
# 실시간 출력을 위한 import
from langchain_teddynote.messages import stream_response

In [18]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field


llm = ChatOpenAI(temperature=0, model_name="gpt-4.1-mini")

In [19]:
email_conversation = """From: 김철수 (chulsoo.kim@bikecorporation.me)
To: 이은채 (eunchae@teddyinternational.me)
Subject: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안

안녕하세요, 이은채 대리님,

저는 바이크코퍼레이션의 김철수 상무입니다. 최근 보도자료를 통해 귀사의 신규 자전거 "ZENESIS"에 대해 알게 되었습니다. 바이크코퍼레이션은 자전거 제조 및 유통 분야에서 혁신과 품질을 선도하는 기업으로, 이 분야에서의 장기적인 경험과 전문성을 가지고 있습니다.

ZENESIS 모델에 대한 상세한 브로슈어를 요청드립니다. 특히 기술 사양, 배터리 성능, 그리고 디자인 측면에 대한 정보가 필요합니다. 이를 통해 저희가 제안할 유통 전략과 마케팅 계획을 보다 구체화할 수 있을 것입니다.

또한, 협력 가능성을 더 깊이 논의하기 위해 다음 주 화요일(1월 15일) 오전 10시에 미팅을 제안합니다. 귀사 사무실에서 만나 이야기를 나눌 수 있을까요?

감사합니다.

김철수
상무이사
바이크코퍼레이션"""

In [20]:
from itertools import chain
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "다음의 이메일 내용중 중요한 내용을 추출해 주세요.\n\n{email_conversation}"
)

llm = ChatOpenAI(temperature=0, model_name="gpt-4.1-mini")

chain = prompt | llm

answer = chain.stream({"email_conversation": email_conversation})

output = stream_response(answer, return_output=True)

중요 내용 요약:

- 김철수 상무(바이크코퍼레이션)가 이은채 대리(테디인터내셔널)에 "ZENESIS" 자전거 유통 협력 제안
- ZENESIS 자전거의 상세 브로슈어 요청 (기술 사양, 배터리 성능, 디자인 정보 포함)
- 유통 전략 및 마케팅 계획 구체화를 위해 정보 필요
- 1월 15일 화요일 오전 10시에 미팅 제안 (귀사 사무실에서)

In [21]:
print(output)

중요 내용 요약:

- 김철수 상무(바이크코퍼레이션)가 이은채 대리(테디인터내셔널)에 "ZENESIS" 자전거 유통 협력 제안
- ZENESIS 자전거의 상세 브로슈어 요청 (기술 사양, 배터리 성능, 디자인 정보 포함)
- 유통 전략 및 마케팅 계획 구체화를 위해 정보 필요
- 1월 15일 화요일 오전 10시에 미팅 제안 (귀사 사무실에서)


In [22]:
class EmailSummary(BaseModel):
    person: str = Field(description="메일을 보낸 사람")
    email: str = Field(description="메일을 보낸 사람의 이메일 주소")
    subject: str = Field(description="메일 제목")
    summary: str = Field(description="메일 본문을 요약한 텍스트")
    date: str = Field(description="메일 본문에 언급된 미팅 날짜와 시간")


# PydanticOutputParser 생성
parser = PydanticOutputParser(pydantic_object=EmailSummary)

In [23]:
# instruction 을 출력합니다.
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"person": {"description": "메일을 보낸 사람", "title": "Person", "type": "string"}, "email": {"description": "메일을 보낸 사람의 이메일 주소", "title": "Email", "type": "string"}, "subject": {"description": "메일 제목", "title": "Subject", "type": "string"}, "summary": {"description": "메일 본문을 요약한 텍스트", "title": "Summary", "type": "string"}, "date": {"description": "메일 본문에 언급된 미팅 날짜와 시간", "title": "Date", "type": "string"}}, "required": ["person", "email", "subject", "summary", "date"]}
```


In [27]:
prompt = PromptTemplate.from_template(
    """
You are a helpful assistant. Please answer the following questions in KOREAN.

QUESTION:
{question}

EMAIL CONVERSATION:
{email_conversation}

FORMAT:
{format}
"""
)
prompt = prompt.partial(format=parser.get_format_instructions())

In [28]:
# chain 을 생성합니다.
chain = prompt | llm

In [29]:
# chain 을 실행하고 결과를 출력합니다.
response = chain.stream(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용중 주요 내용을 추출해 주세요.",
    }
)

# 결과는 JSON 형태로 출력됩니다.
output = stream_response(response, return_output=True)

```json
{
  "person": "김철수",
  "email": "chulsoo.kim@bikecorporation.me",
  "subject": "\"ZENESIS\" 자전거 유통 협력 및 미팅 일정 제안",
  "summary": "바이크코퍼레이션 김철수 상무가 ZENESIS 자전거의 상세 브로슈어(기술 사양, 배터리 성능, 디자인) 요청과 유통 및 마케팅 협력 가능성 논의를 위해 미팅을 제안함.",
  "date": "1월 15일 오전 10시"
}
```

In [30]:
# PydanticOutputParser 를 사용하여 결과를 파싱합니다.
structured_output = parser.parse(output)
print(structured_output)

person='김철수' email='chulsoo.kim@bikecorporation.me' subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안' summary='바이크코퍼레이션 김철수 상무가 ZENESIS 자전거의 상세 브로슈어(기술 사양, 배터리 성능, 디자인) 요청과 유통 및 마케팅 협력 가능성 논의를 위해 미팅을 제안함.' date='1월 15일 오전 10시'


In [31]:
# 출력 파서를 추가하여 전체 체인을 재구성합니다.
chain = prompt | llm | parser

In [32]:
# chain 을 실행하고 결과를 출력합니다.
response = chain.invoke(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용중 주요 내용을 추출해 주세요.",
    }
)

# 결과는 EmailSummary 객체 형태로 출력됩니다.
response

EmailSummary(person='김철수', email='chulsoo.kim@bikecorporation.me', subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안', summary='바이크코퍼레이션 김철수 상무가 ZENESIS 자전거의 상세 브로슈어(기술 사양, 배터리 성능, 디자인)를 요청하며, 유통 협력과 마케팅 계획을 구체화하기 위해 미팅을 제안함.', date='1월 15일 오전 10시')

In [33]:

llm_with_structered = ChatOpenAI(
    temperature=0, model_name="gpt-4.1-mini"
).with_structured_output(EmailSummary)

In [34]:
# invoke() 함수를 호출하여 결과를 출력합니다.
answer = llm_with_structered.invoke(email_conversation)
answer

EmailSummary(person='김철수', email='chulsoo.kim@bikecorporation.me', subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안', summary='김철수 상무는 이은채 대리에게 ZENESIS 자전거에 대한 상세 브로슈어 요청과 함께, 기술 사양, 배터리 성능, 디자인 정보가 필요하다고 전달했습니다. 또한, 1월 15일 화요일 오전 10시에 미팅을 제안하며 협력 가능성을 논의하고자 합니다.', date='2024-01-08')